### GERADOR DE DATASET (RF01)


In [ ]:
##importando bibliotecas

import csv
import json
import re
import os
import random
from datetime import datetime, timedelta
from collections import defaultdict

: 

In [ ]:
# função para gerar um dataset

def gerar_dataset_vendas(caminho_csv="vendas.csv", n_registros=200, seed=42):
    """Gera um dataset sintetico de vendas com dados sujos e grava em CSV."""
    random.seed(seed)
    produtos = ["Notebook", "Smartphone", "Tablet", "Monitor",
                "Teclado", "Mouse", "Headset"]
    categorias = {"Notebook": "Computadores", "Smartphone": "Celulares",
                  "Tablet": "Celulares", "Monitor": "Computadores",
                  "Teclado": "Perifericos", "Mouse": "Perifericos",
                  "Headset": "Perifericos"}
    precos = {"Notebook": 3500, "Smartphone": 2200, "Tablet": 1800,
              "Monitor": 1200, "Teclado": 250, "Mouse": 120,
              "Headset": 350}
    regioes = ["Sudeste", "Sul", "Nordeste", "Centro-Oeste", "Norte"]
    data_inicio = datetime(2025, 1, 1)
    colunas = ["id_venda", "data_venda", "cliente", "produto",
               "categoria", "regiao", "quantidade", "preco_unitario"]

    with open(caminho_csv, "w", newline="", encoding="utf-8") as arquivo:
        escritor = csv.DictWriter(arquivo, fieldnames=colunas)
        escritor.writeheader()

        for i in range(n_registros):
            produto = random.choice(produtos)
            categoria = categorias[produto]
            quantidade = random.randint(1, 10)
            preco = round(precos[produto] * random.uniform(0.85, 1.15), 2)
            data = data_inicio + timedelta(days=random.randint(0, 364))
            data_txt = data.strftime("%Y-%m-%d")
            cliente = f"Cliente_{random.randint(1, 50):03d}"

            # --- sujeira proposital para a etapa de limpeza ---
            if random.random() < 0.05:
                quantidade = ""
            if random.random() < 0.04:
                preco = ""
            if random.random() < 0.06:
                produto = " " + produto + " "
            if random.random() < 0.03:
                data_txt = "DATA INVALIDA"
            if random.random() < 0.10:
                cliente = random.choice([
                    cliente.upper().replace("_", "-"),
                    cliente + "!!",
                    " " + cliente,
                    cliente.replace("Cliente_", "cliente#"),
                ])

            escritor.writerow({
                "id_venda": i + 1,
                "data_venda": data_txt,
                "cliente": cliente,
                "produto": produto,
                "categoria": categoria,
                "regiao": random.choice(regioes),
                "quantidade": quantidade,
                "preco_unitario": preco,
            })

    print(f"Dataset gerado com {n_registros} registros em {caminho_csv}.")

In [ ]:
#se ja existir um dataset chamado vendas.csv essa verificação pula etapa geração de csv

if not os.path.exists("vendas.csv"):
    gerar_dataset_vendas("vendas.csv")
else:
    print("vendas.csv já existe, geração pulada.")

### CARREGANDO O DATASET (RF01)

In [ ]:
def carregar_dataset(caminho_csv):
    """
    Carrega um dataset de vendas a partir de um arquivo CSV.
    Parâmetros:
        caminho_csv (str): caminho para o arquivo CSV.
    Retorna:
        list: lista de dicionários, um por registro.
    """
    try:
        with open(caminho_csv, "r", encoding="utf-8") as arquivo:
            leitor = csv.DictReader(arquivo)
            registros = list(leitor)
        print(f"Arquivo '{caminho_csv}' carregado com sucesso.")
        return registros
    except FileNotFoundError:
        print(f"Erro: o arquivo '{caminho_csv}' não foi encontrado.")
        return []

registros = carregar_dataset("vendas.csv")

print()
print(f"Total de registros carregados: {len(registros)}")
print()
print(registros[0])

### INSPECIONANDO E DESCREVENDO OS DADOS (RF02)

In [ ]:
def inspecionar_dados(registros):
    """
    Exibe as informações estruturais do dataset carregado:
    total de registros, colunas, valores ausentes por coluna
    e os primeiros registros.
    Parâmetros:
        registros (list): lista de dicionários representando o dataset.
    Retorna:
        list: os mesmos registros recebidos (sem alteração).
    """
    if not registros:
        print("O dataset está vazio.")
        return registros

    total = len(registros)
    colunas = list(registros[0].keys())

    nulos = {coluna: 0 for coluna in colunas}
    for linha in registros:
        for coluna in colunas:
            if linha[coluna] == "":
                nulos[coluna] += 1

    print("=== INSPEÇÃO INICIAL DO DATASET ===")
    print()
    print(f"Total de registros: {total}")
    print(f"Total de colunas: {len(colunas)}")
    print(f"\nColunas: {colunas}")
    print(f"\nValores ausentes por coluna:\n{nulos}")
    print("\nPrimeiros 5 registros:")
    for linha in registros[:5]:
        print(linha)

    return registros

registros = inspecionar_dados(registros)

### LIMPANDO E TRATANDO OS DADOS (RF03)

In [ ]:
def limpar_dados(dataset):
    """ Limpa os dados do dataset, removendo espaços em branco, convertendo tipos e tratando valores ausentes.
    Parâmetros:
        dataset (list): lista de dicionários com os registros.
    Retorna:
        tupla: (lista de dicionários limpos, relatório de limpeza) """

    relatorio = {
        'iniciais': len(dataset),
        'removidos_data': 0,
        'removidos_nulos': 0,
        'clientes_padronizados': 0,
        'finais': 0
    }

    padrao_cliente = re.compile(r"^Cliente_\d{3}$", flags=re.IGNORECASE)
    registros_limpos = []

    for linha in dataset:
        linha = dict(linha)  # cópia: não altera os registros originais

        # 1. Remover espaços extras nas colunas de texto
        for chave in ["cliente", "produto", "categoria", "regiao"]:
            linha[chave] = linha[chave].strip()

        # 2. Conversão de "data_venda" e descartar datas inválidas
        try:
            linha['data_venda'] = datetime.strptime(linha['data_venda'], "%Y-%m-%d")
        except ValueError:
            relatorio['removidos_data'] += 1
            continue

        # 3. Descartar nulos em "quantidade" e "preco_unitario"
        if linha['quantidade'] == "" or linha['preco_unitario'] == "":
            relatorio['removidos_nulos'] += 1
            continue

        # 4. Ajustar os tipos numéricos
        linha['quantidade'] = int(float(linha['quantidade']))
        linha['preco_unitario'] = float(linha['preco_unitario'])

        # 5. Padronizar "nome do cliente" para o formato Cliente_NNN
        #    - sinaliza quem veio fora do padrão (ex.: "cliente#016", "CLIENTE-016", "Cliente_008!!")
        #    - remove caracteres não alfanuméricos com re.sub()
        #    - reconstrói o nome no formato Cliente_NNN a partir dos 3 dígitos finais
        linha["cliente_fora_do_padrao"] = padrao_cliente.match(linha['cliente']) is None

        nome_limpo = re.sub(r"[^A-Za-z0-9]", "", linha['cliente'])
        numero = re.search(r"\d{3}$", nome_limpo)
        if numero:
            linha['cliente'] = f"Cliente_{numero.group()}"
        else:
            linha['cliente'] = nome_limpo

        if linha["cliente_fora_do_padrao"]:
            relatorio['clientes_padronizados'] += 1

        registros_limpos.append(linha)

    # 6. Montar e imprimir relatório de limpeza
    relatorio["finais"] = len(registros_limpos)
    print("=== RELATÓRIO DE LIMPEZA ===")
    print()
    print(relatorio)
    return registros_limpos, relatorio

In [ ]:
registros_limpos, relatorio = limpar_dados(registros)